# 📡 Advanced Forecasting
**fpppy Chapters 10–12 · Pattern Recognition for the Rest of Us**

> Prophet handles complex seasonality and holidays with minimal tuning. LSTMs learn temporal patterns end-to-end. Time series cross-validation gives honest error estimates. This notebook covers the modern toolkit.

### Required reading
📘 [fpppy Ch 10–12](https://otexts.com/fpppy/) · [Prophet docs](https://facebook.github.io/prophet/)

### What you'll learn
- Prophet — Facebook's decomposable time series model
- Time series cross-validation (no data leakage!)
- Feature engineering for ML-based forecasting
- LSTM for sequence modeling (conceptual + implementation)
- Choosing between ARIMA, ETS, Prophet, and ML approaches

### Dataset: Bitcoin price (complex seasonality) + Retail
### Time: ~70 minutes

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
!pip install prophet -q
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("✓ Prophet installed")

In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os

# Clone the course repo if not already present (works in Colab)
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    # Fallback: already in repo root (e.g. running locally)
    DATA_DIR = '../data'

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Available datasets: {os.listdir(DATA_DIR)}")

### Load Dataset: Temperatures

In [ ]:
import pandas as pd
temps_df = pd.read_csv(f'{DATA_DIR}/Temperatures.csv', parse_dates=['Date'])
temps_df = temps_df.rename(columns={'Date':'ds','Temp':'y'})
print(f"Temperatures: {temps_df.shape}  ({temps_df['ds'].min().date()} to {temps_df['ds'].max().date()})")
temps_df.head()

In [ ]:
# Melbourne daily temperatures — hardcoded sample + synthetic extension
import pandas as pd, numpy as np
np.random.seed(42)
# 10 years of synthetic seasonal temperature data (Melbourne-like)
dates = pd.date_range('2000-01-01', periods=3650, freq='D')
# Annual cycle: cold July (~5°C), warm Jan (~18°C)
day_of_year = np.array([d.timetuple().tm_yday for d in dates])
seasonal = 11.5 + 6.5 * np.cos(2 * np.pi * (day_of_year - 15) / 365)
noise = np.random.normal(0, 2, len(dates))
temps_data = pd.DataFrame({'ds': dates, 'y': seasonal + noise})
temps_data['y'] = temps_data['y'].clip(0, 35)
print(f"Temperature dataset: {temps_data.shape}")
temps_data.head()

## 🔮 Part 1 — Prophet: Decomposable Forecasting

Prophet models time series as:
```
y(t) = trend(t) + seasonality(t) + holidays(t) + ε
```

**Why Prophet?**
- Handles multiple seasonalities (daily, weekly, yearly) automatically
- Handles missing data and outliers gracefully  
- Allows domain knowledge injection (holidays, capacity limits)
- Uncertainty intervals included by default
- Minimal tuning — works well out of the box

In [ ]:
# Fit Prophet model
train_size = int(len(temps) * 0.85)
train_df = temps.iloc[:train_size]
test_df  = temps.iloc[train_size:]

model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,  # daily temps don't have weekly patterns
    daily_seasonality=False,
    changepoint_prior_scale=0.05,  # flexibility of trend changes
    interval_width=0.95
)
model.fit(train_df)

# Forecast
future = model.make_future_dataframe(periods=len(test_df))
forecast = model.predict(future)

print("✓ Prophet model fitted")
print(f"Forecast columns: {[c for c in forecast.columns if c in ['ds','yhat','yhat_lower','yhat_upper','trend','yearly']]}")

In [ ]:
# Plot forecast
fig1 = model.plot(forecast, figsize=(13, 5))
fig1.axes[0].set_title('Prophet Forecast — Melbourne Daily Min Temperature')
fig1.axes[0].axvline(test_df['ds'].iloc[0], color='#e85d20', lw=1.5, ls='--', label='Train/Test split')
plt.tight_layout()
plt.show()

# Components
fig2 = model.plot_components(forecast, figsize=(12, 7))
fig2.suptitle('Prophet Components: Trend + Seasonality', y=1.01)
plt.tight_layout()
plt.show()

# Evaluate on test set
test_forecast = forecast[forecast['ds'].isin(test_df['ds'])]
mae  = mean_absolute_error(test_df['y'], test_forecast['yhat'])
rmse = np.sqrt(mean_squared_error(test_df['y'], test_forecast['yhat']))
print(f"\nProphet test performance:")
print(f"  MAE:  {mae:.3f}°C")
print(f"  RMSE: {rmse:.3f}°C")

## ⏱️ Part 2 — Time Series Cross-Validation

Regular K-fold CV would leak future data into training. Time series CV uses **expanding windows** (or sliding windows):

```
Fold 1: Train [1..t]     → Validate [t+1..t+h]
Fold 2: Train [1..t+h]   → Validate [t+h+1..t+2h]
...
```

Never validate on data that comes before your training data ends.

In [ ]:
# Prophet built-in cross-validation
print("Running Prophet time series cross-validation...")
df_cv = cross_validation(
    model,
    initial='1000 days',   # minimum training window
    period='180 days',     # how often to re-fit
    horizon='365 days',    # forecast horizon
    parallel=None
)
df_perf = performance_metrics(df_cv)

print("\n=== Cross-Validation Performance by Horizon ===")
print(df_perf[['horizon','mae','rmse','mape']].head(10).to_string(index=False))

fig = plot_cross_validation_metric(df_cv, metric='rmse', figsize=(10,4))
plt.title('Prophet CV: RMSE by Forecast Horizon\n(error grows with horizon — expected)')
plt.tight_layout()
plt.show()
print("\n📌 Errors grow as we forecast further into the future — uncertainty compounds")

## 🧠 Part 3 — ML-based Forecasting: Feature Engineering

Classical models (ARIMA, ETS) use autocorrelation directly. ML models (Random Forest, XGBoost, LSTM) need **lag features** — create columns for t-1, t-7, t-28, etc.

This converts the time series problem into a standard supervised learning problem.

In [ ]:
# Feature engineering for ML forecasting
df_feat = temps.copy()
df_feat = df_feat.set_index('ds')
df_feat.index.name = 'date'

# Create lag features
for lag in [1, 2, 3, 7, 14, 28, 365]:
    df_feat[f'lag_{lag}'] = df_feat['y'].shift(lag)

# Rolling statistics
df_feat['rolling_7_mean']  = df_feat['y'].rolling(7).mean()
df_feat['rolling_30_mean'] = df_feat['y'].rolling(30).mean()
df_feat['rolling_7_std']   = df_feat['y'].rolling(7).std()

# Calendar features
df_feat['month']      = df_feat.index.month
df_feat['dayofyear']  = df_feat.index.dayofyear
df_feat['quarter']    = df_feat.index.quarter

df_feat = df_feat.dropna()
print(f"Features after engineering: {df_feat.shape[1]-1} predictors")
print(f"Sample features: {list(df_feat.columns)}")

In [ ]:
# Train XGBoost on lag features
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit

feature_cols = [c for c in df_feat.columns if c != 'y']
X_ml = df_feat[feature_cols].values
y_ml = df_feat['y'].values

# Time series split
n = len(X_ml)
split = int(n * 0.85)
X_tr_ml, X_te_ml = X_ml[:split], X_ml[split:]
y_tr_ml, y_te_ml = y_ml[:split], y_ml[split:]

gbm = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
gbm.fit(X_tr_ml, y_tr_ml)
y_pred_ml = gbm.predict(X_te_ml)

mae_ml  = mean_absolute_error(y_te_ml, y_pred_ml)
rmse_ml = np.sqrt(mean_squared_error(y_te_ml, y_pred_ml))
print(f"ML (GBM + lag features) test performance:")
print(f"  MAE:  {mae_ml:.3f}°C")
print(f"  RMSE: {rmse_ml:.3f}°C")

# Feature importance
imp = pd.Series(gbm.feature_importances_, index=feature_cols).sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(imp.index, imp.values, color='#1e5fa8', edgecolor='white')
ax.set_ylabel('Feature Importance')
ax.set_title('GBM Forecasting — Top 10 Features')
ax.set_xticklabels(imp.index, rotation=45, ha='right')
plt.tight_layout()
plt.show()
print("\n📌 lag_365 (same day last year) is the most important feature — strong annual seasonality")
print("   This is just the ML equivalent of seasonal naive, but combined with other patterns")

## 📊 Part 4 — Model Selection Guide

| Model | Best for | Weaknesses |
|-------|----------|------------|
| **Naive/SNaive** | Baseline, sanity check | Can't improve with data |
| **ETS/Holt-Winters** | Short series, stable seasonality | Limited by assumed structure |
| **ARIMA** | Stationary-after-diff, complex autocorrelation | Manual order selection |
| **Prophet** | Business data, multiple seasonalities, holidays | Slower, less transparent |
| **ML + lags** | Large datasets, nonlinear patterns, many covariates | Needs many observations, can leak |
| **LSTM** | Very long sequences, complex dependencies | Data hungry, hard to tune |

In [ ]:
answers={"q1":"","q2":"","q3":"","q4":"","q5":""}
# q1: What is the key difference between Prophet and ARIMA?
# q2: Why can't you use regular K-fold CV for time series?
# q3: How do you convert a time series problem into a supervised ML problem?
# q4: What is changepoint_prior_scale in Prophet and what does it control?
# q5: Name one scenario where you'd choose ARIMA over Prophet
missing=[k for k,v in answers.items() if not v.strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ Done! File → Save a copy in GitHub")

### 📤 Submit your results

In [ ]:
# ── Submit your results to the instructor ────────────────────────────────────
# Fill in your GitHub username (do this once, it stays saved in the notebook)
GITHUB_USERNAME = ""   # ← put your GitHub username here e.g. "jsmith42"

# ─────────────────────────────────────────────────────────────────────────────
# DO NOT EDIT BELOW THIS LINE
import json, urllib.parse

def build_submission_url(username, notebook_name, score, total, answers_dict,
                          form_base_url=None, entries=None):
    """Build a pre-filled Google Form URL for quiz submission."""
    if not form_base_url:
        print("⚠  No submission URL configured yet.")
        print("   Your instructor will share the Form URL — paste it into form_base_url below.")
        print("   Your answers are saved in this notebook regardless.")
        return None

    params = {
        entries.get('user',     'entry.000000001'): username,
        entries.get('notebook', 'entry.000000002'): notebook_name,
        entries.get('score',    'entry.000000003'): str(score),
        entries.get('total',    'entry.000000004'): str(total),
    }
    url = form_base_url.replace('/viewform','') + '/viewform?' + urllib.parse.urlencode(params)
    return url

# ── Submission config (your instructor fills these in) ───────────────────────
FORM_BASE_URL = None   # e.g. "https://docs.google.com/forms/d/e/XXXX/viewform"
FORM_ENTRIES  = {      # entry IDs from the Google Form
    'user':     'entry.000000001',
    'notebook': 'entry.000000002',
    'score':    'entry.000000003',
    'total':    'entry.000000004',
}
# ─────────────────────────────────────────────────────────────────────────────

# Calculate score from answers dict
score_val = sum(1 for v in answers.values() if str(v).strip())  # counts answered Qs
# For auto-graded notebooks the score is passed directly
# Here we just verify completion
n_answered = sum(1 for v in answers.values() if str(v).strip())
n_total    = len(answers)

print(f"\n{'='*50}")
print(f"Quiz completion: {n_answered}/{n_total} questions answered")
if n_answered < n_total:
    print(f"⚠  Please answer all {n_total} questions before submitting.")
else:
    print("✅ All questions answered!")
    if GITHUB_USERNAME:
        import os
        nb_name = os.path.basename(globals().get('__vsc_ipynb_file__',
                  globals().get('__file__', 'unknown-notebook'))).replace('.ipynb','')
        url = build_submission_url(GITHUB_USERNAME, nb_name,
                                   n_answered, n_total, answers,
                                   FORM_BASE_URL, FORM_ENTRIES)
        if url:
            print(f"\n🔗 Submit to instructor:")
            print(f"   {url}")
            print("\n   Copy the URL above, open it in a new tab, and click Submit.")
        else:
            print("\n   Save your notebook to GitHub to record your progress:")
            print("   File → Save a copy in GitHub → choose your fork")
    else:
        print("\n⚠  Add your GitHub username to GITHUB_USERNAME above, then re-run.")
    print(f"{'='*50}")